# 功能测试手册

在 Jupyter Notebook 中逐 Cell 执行，每个 Cell 独立可运行。
建议按顺序执行，Cell 1 ~ 2 是前置检查。

## Cell 1：环境检查

In [ ]:
import sys
print(f"Python: {sys.version.split()[0]}")

packages = {
    "cv2": "opencv",
    "mediapipe": "mediapipe",
    "numpy": "numpy",
    "aitoolkit_cam": "aitoolkit_cam",
    "esp32_arduino": "esp32_arduino",
    "requests": "requests",
}
for mod, name in packages.items():
    try:
        m = __import__(mod)
        ver = getattr(m, "__version__", "ok")
        print(f"  ✅ {name}: {ver}")
    except ImportError as e:
        print(f"  ❌ {name}: {e}")

import os, glob
videos = sorted([f for f in os.listdir("/dev") if f.startswith("video")])
ports  = glob.glob("/dev/ttyUSB*") + glob.glob("/dev/ttyACM*")
print(f"
摄像头设备: {videos}")
print(f"串口设备:   {ports if ports else '未找到'}")

## Cell 2：摄像头探测（确定可用的 source 编号）

In [ ]:
import cv2, os

CAMERA_SOURCE = None
for idx in [5, 6, 0, 1, 2]:
    path = f"/dev/video{idx}"
    if not os.path.exists(path):
        print(f"❌ video{idx} 设备不存在")
        continue
    cap = cv2.VideoCapture(path, cv2.CAP_V4L2)
    if cap.isOpened():
        ret, frame = cap.read()
        cap.release()
        if ret and frame is not None:
            print(f"✅ video{idx} 可用，分辨率: {frame.shape[1]}x{frame.shape[0]}")
            CAMERA_SOURCE = idx
            break
        else:
            print(f"⚠️  video{idx} 打开但读不到帧")
    else:
        cap.release()
        print(f"❌ video{idx} 无法打开")

if CAMERA_SOURCE is None:
    print("⚠️  未找到可用摄像头，后续 Cell 将无法运行")
else:
    print(f"
后续测试使用: source={CAMERA_SOURCE}")

## Cell 3：基础推流

前提：Cell 2 通过

In [ ]:
from aitoolkit_cam import Camera

cam = Camera(source=CAMERA_SOURCE, max_frames=100)
url = cam.start()
print(f"视频流地址: {url}")
print("推送 100 帧原始画面...")

for frame in cam:
    cam.show(frame)

cam.stop()
print("✅ 基础流测试完成")

## Cell 4：OpenCV 处理 — 边缘检测 + 帧率显示

前提：Cell 2 通过

In [ ]:
import cv2, time
from aitoolkit_cam import Camera

cam = Camera(source=CAMERA_SOURCE, max_frames=150)
url = cam.start()
print(f"视频流: {url}")

frame_count = 0
t0 = time.time()

for frame in cam:
    gray   = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges  = cv2.Canny(gray, 50, 150)
    result = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

    frame_count += 1
    fps = frame_count / (time.time() - t0 + 1e-6)
    cv2.putText(result, f"FPS: {fps:.1f}  Frame: {frame_count}",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)
    cam.show(result)

cam.stop()
print(f"✅ 完成，平均 FPS: {fps:.1f}")

## Cell 5：MediaPipe 姿态检测

前提：Cell 2 通过

In [ ]:
import cv2, mediapipe as mp
from aitoolkit_cam import Camera

mp_drawing = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(min_detection_confidence=0.5,
                                 min_tracking_confidence=0.5)

KEYPOINT_NAMES = [
    "鼻子","左眼内","左眼","左眼外","右眼内","右眼","右眼外",
    "左耳","右耳","嘴左","嘴右","左肩","右肩","左肘","右肘",
    "左手腕","右手腕","左髋","右髋","左膝","右膝","左脚踝","右脚踝"
]

def draw_pose(frame):
    h, w = frame.shape[:2]
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    rgb.flags.writeable = False
    results = holistic.process(rgb)
    rgb.flags.writeable = True

    if results.pose_landmarks:
        mp_drawing.draw_landmarks(
            frame, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(0,255,0), thickness=2, circle_radius=3),
            mp_drawing.DrawingSpec(color=(0,200,0), thickness=2))
        for i, lm in enumerate(results.pose_landmarks.landmark):
            if i < len(KEYPOINT_NAMES) and lm.visibility > 0.5:
                cv2.putText(frame, KEYPOINT_NAMES[i],
                            (int(lm.x*w), int(lm.y*h)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.35,
                            (255,255,255), 1, cv2.LINE_AA)

    if results.face_landmarks:
        mp_drawing.draw_landmarks(
            frame, results.face_landmarks, mp_holistic.FACEMESH_CONTOURS,
            mp_drawing.DrawingSpec(color=(255,80,80), thickness=1, circle_radius=1),
            mp_drawing.DrawingSpec(color=(255,80,80), thickness=1))

    for lms, color in [
        (results.left_hand_landmarks,  (80,80,255)),
        (results.right_hand_landmarks, (80,255,255))
    ]:
        if lms:
            mp_drawing.draw_landmarks(
                frame, lms, mp_holistic.HAND_CONNECTIONS,
                mp_drawing.DrawingSpec(color=color, thickness=2, circle_radius=3),
                mp_drawing.DrawingSpec(color=color, thickness=2))

    parts = []
    if results.pose_landmarks:       parts.append("身体")
    if results.face_landmarks:       parts.append("面部")
    if results.left_hand_landmarks:  parts.append("左手")
    if results.right_hand_landmarks: parts.append("右手")
    cv2.putText(frame, " | ".join(parts) if parts else "未检测到",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
    return frame

cam = Camera(source=CAMERA_SOURCE, max_frames=200)
url = cam.start()
print(f"视频流: {url}")

for frame in cam:
    cam.show(draw_pose(frame))

holistic.close()
cam.stop()
print("✅ 姿态检测完成")

## Cell 6：手势识别 — 数手指

前提：Cell 2 通过

In [ ]:
import cv2, mediapipe as mp
from aitoolkit_cam import Camera

mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(max_num_hands=2,
                        min_detection_confidence=0.7,
                        min_tracking_confidence=0.5)

FINGERTIPS   = [4, 8, 12, 16, 20]
FINGER_NAMES = ["拇指","食指","中指","无名指","小指"]

def count_fingers(lms, handedness):
    lm = lms.landmark
    fingers = []
    fingers.append(1 if (handedness == "Right" and lm[4].x < lm[3].x) or
                        (handedness == "Left"  and lm[4].x > lm[3].x) else 0)
    for tip in FINGERTIPS[1:]:
        fingers.append(1 if lm[tip].y < lm[tip-2].y else 0)
    return fingers

def draw_hands(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    rgb.flags.writeable = False
    results = hands.process(rgb)
    rgb.flags.writeable = True

    if results.multi_hand_landmarks:
        for i, (lms, info) in enumerate(
            zip(results.multi_hand_landmarks, results.multi_handedness)
        ):
            side = info.classification[0].label
            mp_drawing.draw_landmarks(
                frame, lms, mp_hands.HAND_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(80,80,255), thickness=2, circle_radius=4),
                mp_drawing.DrawingSpec(color=(80,200,255), thickness=2))

            fingers = count_fingers(lms, side)
            up = [FINGER_NAMES[j] for j, f in enumerate(fingers) if f]
            y = 30 + i * 80
            cv2.putText(frame, f"{side}: {sum(fingers)} 根手指",
                        (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)
            cv2.putText(frame, " ".join(up) if up else "握拳",
                        (10, y+30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1)
    return frame

cam = Camera(source=CAMERA_SOURCE, max_frames=200)
url = cam.start()
print(f"视频流: {url}")
print("伸出手指，显示几根手指朝上...")

for frame in cam:
    cam.show(draw_hands(frame))

hands.close()
cam.stop()
print("✅ 手势识别完成")

## Cell 7：人脸检测 + 眼部打码

前提：Cell 2 通过

In [ ]:
import cv2, numpy as np
from aitoolkit_cam import Camera

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

def mosaic(frame, x, y, w, h, scale=0.1):
    roi   = frame[y:y+h, x:x+w]
    small = cv2.resize(roi, (max(1, int(w*scale)), max(1, int(h*scale))))
    frame[y:y+h, x:x+w] = cv2.resize(small, (w, h),
                                       interpolation=cv2.INTER_NEAREST)

def detect_faces(frame):
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(60,60))
    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x,y), (x+w,y+h), (0,255,0), 2)
        cv2.putText(frame, f"Face {w}x{h}", (x, y-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
        mosaic(frame, x, y + h//5, w, h//4)   # 眼睛区域打码
    cv2.putText(frame, f"Faces: {len(faces)}",
                (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)
    return frame

cam = Camera(source=CAMERA_SOURCE, max_frames=150)
url = cam.start()
print(f"视频流: {url}")

for frame in cam:
    cam.show(detect_faces(frame))

cam.stop()
print("✅ 人脸检测完成")

## Cell 8：颜色追踪 — 追踪红色物体

前提：Cell 2 通过  
对着摄像头展示红色物体（红笔、红本子等）

In [ ]:
import cv2, numpy as np
from aitoolkit_cam import Camera

def track_color(frame):
    hsv   = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask1 = cv2.inRange(hsv, np.array([0,100,100]),   np.array([10,255,255]))
    mask2 = cv2.inRange(hsv, np.array([160,100,100]), np.array([180,255,255]))
    mask  = cv2.morphologyEx(cv2.bitwise_or(mask1, mask2),
                             cv2.MORPH_OPEN,  np.ones((5,5), np.uint8))
    mask  = cv2.dilate(mask, np.ones((5,5), np.uint8))

    for cnt in cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)[0]:
        if cv2.contourArea(cnt) < 500:
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        cx, cy = x+w//2, y+h//2
        cv2.rectangle(frame, (x,y), (x+w,y+h), (0,0,255), 2)
        cv2.circle(frame, (cx,cy), 5, (0,0,255), -1)
        cv2.putText(frame, f"Red {int(cv2.contourArea(cnt))}px",
                    (x, y-8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)

    hf, wf = frame.shape[:2]
    thumb = cv2.resize(cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR),
                       (wf//5, hf//5))
    frame[10:10+thumb.shape[0], wf-thumb.shape[1]-10:wf-10] = thumb
    return frame

cam = Camera(source=CAMERA_SOURCE, max_frames=150)
url = cam.start()
print(f"视频流: {url}")

for frame in cam:
    cam.show(track_color(frame))

cam.stop()
print("✅ 颜色追踪完成")

## Cell 9：ESP32 连接测试

前提：USB 线已接好，`/dev/ttyUSB0` 存在

In [ ]:
from esp32_arduino import *

print("尝试连接 ESP32...")
ok = esp32_begin()

if ok:
    print("✅ ESP32 连接成功")
    info = getSystemInfo()
    print(f"库版本: {info['library_version']}")
    print(f"硬件可用: {info['hardware_available']}")
    if _esp32_instance and _esp32_instance.ping():
        print("✅ PING 响应正常")
    esp32_close()
else:
    print("❌ ESP32 未连接，请检查：")
    print("   1. USB 线是否插好")
    print("   2. /dev/ttyUSB0 是否存在")
    print("   3. 串口权限：sudo chmod 666 /dev/ttyUSB0")

## Cell 10：ESP32 传感器读取

前提：Cell 9 通过，DHT11 已接好

In [ ]:
from esp32_arduino import *
import time

if esp32_begin():
    print("✅ 开始读取传感器（每秒一次，共 10 次）
")
    for i in range(10):
        data  = readAllSensors()
        temp  = data['temperature']
        humid = data['humidity']
        bar_t = "█" * int(temp / 2)
        bar_h = "█" * int(humid / 5)
        print(f"[{i+1:2d}] 温度: {temp:5.1f}°C  {bar_t}")
        print(f"      湿度: {humid:5.1f}%   {bar_h}")
        time.sleep(1)
    esp32_close()
    print("
✅ 传感器测试完成")
else:
    print("❌ ESP32 未连接")

## Cell 11：摄像头 + ESP32 联动

前提：Cell 2 + Cell 9 均通过

In [ ]:
import cv2, time, math
from aitoolkit_cam import Camera
from esp32_arduino import *

esp32_ok  = esp32_begin()
last_read = 0
temp, humid = 25.0, 60.0

def get_sensor():
    global last_read, temp, humid
    if time.time() - last_read > 2.0:
        if esp32_ok:
            d     = readAllSensors()
            temp  = d['temperature']
            humid = d['humidity']
        else:
            temp  = 25 + 2 * math.sin(time.time())
            humid = 60 + 5 * math.cos(time.time())
        last_read = time.time()
    return temp, humid

cam = Camera(source=CAMERA_SOURCE, max_frames=200)
url = cam.start()
print(f"视频流: {url}")
print(f"ESP32: {'已连接' if esp32_ok else '模拟数据'}")

for frame in cam:
    t, h = get_sensor()
    overlay = frame.copy()
    cv2.rectangle(overlay, (0,0), (280,80), (0,0,0), -1)
    cv2.addWeighted(overlay, 0.5, frame, 0.5, 0, frame)
    cv2.putText(frame, f"Temp:  {t:.1f} C",
                (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,200,255), 2)
    cv2.putText(frame, f"Humid: {h:.1f} %",
                (10,65), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,200), 2)
    cam.show(frame)

cam.stop()
if esp32_ok:
    esp32_close()
print("✅ 联动测试完成")

## Cell 12：AI 功能测试

前提：后端已注入 `MOONSHOT_API_KEY`

In [ ]:
import requests

BASE = "http://127.0.0.1:18001"

print("=== AI 代码解释 ===")
r = requests.post(f"{BASE}/api/v1/chat/code/explain",
                  json={"code": "for i in range(5):
    print(i * i)"})
print(r.json().get("explanation", r.text[:200]) if r.ok else f"❌ {r.status_code}")

print("
=== AI 错误分析 ===")
r = requests.post(f"{BASE}/api/v1/chat/error/analyze",
                  json={
                      "error_type": "NameError",
                      "error_message": "name 'pritn' is not defined",
                      "code_context": "pritn('hello')"
                  })
print(r.json().get("analysis", r.text[:200]) if r.ok else f"❌ {r.status_code}")

## 常见问题

| 现象 | 原因 | 解决 |
|------|------|------|
| `RuntimeError: 摄像头启动失败（设备 5 不可用）` | video5 还没准备好 | 先跑 Cell 2 确认可用编号 |
| `❌ video5 无法打开` | 摄像头未插或被占用 | 重新插拔 USB，检查其他进程是否占用 |
| AI 返回 `401 Invalid Authentication` | API Key 无效 | 检查 `.env` 文件里的 `MOONSHOT_API_KEY` |
| AI 返回 `429 Too Many Requests` | 调用太频繁 | 等几秒再试 |
| ESP32 `❌ 未连接` | 串口不存在或无权限 | `ls /dev/ttyUSB*`，若存在则 `chmod 666` |
| 前端视频画面不动 | WebSocket 未连接 | 点击前端“连接视频”按钮 |